# AI Sentinel - Full Fine-Tuning on Colab

This notebook performs full fine-tuning of the violence detection model.
It is separate from the probe notebook to keep things clean.

**WARNING**: Training is disabled by default. Set `RUN_FULL_TRAINING = True` only after user approval.

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Check Dataset and Manifest

In [ ]:
from pathlib import Path

DATASET_ROOT = Path('/content/drive/MyDrive/violence_data')
MANIFEST = Path('/content/drive/MyDrive/AI-Sentinel/manifests/violence_data_manifest.jsonl')

print('Dataset exists:', DATASET_ROOT.exists())
print('Manifest exists:', MANIFEST.exists())

if MANIFEST.exists():
    lines = MANIFEST.read_text().strip().split('\n')
    print('Manifest entries:', len(lines))
    
    import json
    labels = set(json.loads(l)['label'] for l in lines)
    print('Labels:', labels)

## Step 3: Extract Project Code

In [ ]:
import zipfile, shutil
from pathlib import Path

ZIP_CANDIDATES = [
    Path('/content/drive/MyDrive/AI-Sentinel/code/ai-sentinel-colab-code.zip'),
    Path('/content/drive/MyDrive/AI-Sentinel/code/ai-sentinel-code.zip'),
]

ZIP_PATH = None
for candidate in ZIP_CANDIDATES:
    if candidate.exists():
        ZIP_PATH = candidate
        break

print('ZIP_PATH:', ZIP_PATH)
print('ZIP exists:', ZIP_PATH is not None and ZIP_PATH.exists())

if ZIP_PATH is None or not ZIP_PATH.exists():
    raise FileNotFoundError('Project zip not found in Drive')

PROJECT_DIR = Path('/content/ai-sentinel')
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(PROJECT_DIR)

print('Project extracted to:', PROJECT_DIR)

## Step 4: Install Dependencies

In [ ]:
import importlib.util

required_packages = {
    'torch': 'torch',
    'torchvision': 'torchvision',
    'cv2': 'opencv-python-headless',
    'numpy': 'numpy',
    'yaml': 'pyyaml',
    'pytorchvideo': 'pytorchvideo',
}

missing_packages = [pip_name for mod, pip_name in required_packages.items() if importlib.util.find_spec(mod) is None]

if missing_packages:
    print(f'Installing: {missing_packages}')
    !pip install -q {' '.join(missing_packages)}
    print('Installation complete.')
else:
    print('All packages already installed.')

# Verify
for mod in required_packages:
    status = 'OK' if importlib.util.find_spec(mod) is not None else 'MISSING'
    print(f'{mod}: {status}')

## Step 5: GPU Check

In [ ]:
import torch
import os

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    device = 'cuda'
    gpu_name = torch.cuda.get_device_name(0)
    print('GPU:', gpu_name)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Total GPU memory: {total_mem:.2f} GB')
    recommended_bs = 4 if total_mem >= 16 else (2 if total_mem >= 8 else 1)
else:
    device = 'cpu'
    gpu_name = 'N/A'
    recommended_bs = 2
    print('WARNING: Using CPU - training will be slow!')

print(f'\nRecommended device: {device}')
print(f'Recommended batch size: {recommended_bs}')

os.environ['CUDA_VISIBLE_DEVICES'] = '0' if device == 'cuda' else ''

## Step 6: Training Configuration

In [ ]:
# Training parameters - review before running
TRAINING_CONFIG = {
    'stage': 1,  # Change to 2 or 3 after approval
    'epochs': 1,  # 1 for stage 1, 1 for stage 2, 3 for stage 3
    'batch_size': 2,  # Adjust based on GPU memory
    'learning_rate': 1e-4,  # 1e-4 for stage 1, 3e-5 for stage 2, 1e-5 for stage 3
    'val_ratio': 0.5,  # 0.5 for stages 1-2, 0.2 for stage 3
    'max_samples': 64,  # 64 for stage 1, 256 for stage 2, 0 for full (stage 3)
    'unfreeze_backbone': False,  # True for stages 2-3
}

print('Training Configuration:')
for k, v in TRAINING_CONFIG.items():
    print(f'  {k}: {v}')

## Step 7: Full Fine-Tuning (GUARDED)

In [ ]:
RUN_FULL_TRAINING = False  # Set to True only after user approval

if not RUN_FULL_TRAINING:
    raise RuntimeError(
        'Full training is disabled. '
        'Set RUN_FULL_TRAINING=True only after user approval.'
    )

import os, subprocess, time
from pathlib import Path

os.chdir('/content/ai-sentinel')

env = os.environ.copy()
env['PYTHONPATH'] = 'backend'

CHECKPOINT_DIR = Path('/content/drive/MyDrive/AI-Sentinel/checkpoints/full_finetune/')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    'python', 'backend/train_finetune.py',
    '--sources', '/content/drive/MyDrive/violence_data',
    '--weights', 'backend/best_model.pt',
    '--output', str(CHECKPOINT_DIR / 'best_candidate.pt'),
    '--epochs', str(TRAINING_CONFIG['epochs']),
    '--batch-size', str(TRAINING_CONFIG['batch_size']),
    '--lr', str(TRAINING_CONFIG['learning_rate']),
    '--val-ratio', str(TRAINING_CONFIG['val_ratio']),
    '--seed', '42',
    '--num-workers', '2',
]

if TRAINING_CONFIG['max_samples'] > 0:
    cmd.extend(['--max-samples', str(TRAINING_CONFIG['max_samples'])])

if TRAINING_CONFIG['unfreeze_backbone']:
    cmd.append('--unfreeze-backbone')

print(f'Running: {' '.join(cmd)}')
start = time.time()
result = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=7200)
runtime = time.time() - start

print(f'\nTraining completed in {runtime:.2f}s')
print(f'Return code: {result.returncode}')
print(f'\nSTDOUT (last 4000 chars):\n{result.stdout[-4000:]}')
if result.stderr:
    print(f'\nSTDERR (last 4000 chars):\n{result.stderr[-4000:]}')

if result.returncode != 0:
    raise RuntimeError('Training failed. Check stderr above.')

## Step 8: Parse Results

In [ ]:
from pathlib import Path
import json

CHECKPOINT_DIR = Path('/content/drive/MyDrive/AI-Sentinel/checkpoints/full_finetune/')

# Find latest checkpoint
all_checkpoints = sorted(
    CHECKPOINT_DIR.rglob('best_candidate.pt'),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if all_checkpoints:
    latest = all_checkpoints[0]
    print('Latest checkpoint:', latest)
    
    import torch
    checkpoint = torch.load(latest, map_location='cpu')
    
    print('\nCheckpoint metrics:')
    if 'metrics' in checkpoint:
        print(json.dumps(checkpoint['metrics'], indent=2))
    
    print('\nTraining args:')
    if 'trainArgs' in checkpoint:
        print(json.dumps(checkpoint['trainArgs'], indent=2))
    
    print('\nHistory (last 3 epochs):')
    if 'history' in checkpoint:
        for epoch_data in checkpoint['history'][-3:]:
            print(json.dumps(epoch_data))
else:
    print('No checkpoints found.')

## Step 9: Save Report

In [ ]:
from pathlib import Path
import json
from datetime import datetime

report_dir = Path('/content/drive/MyDrive/AI-Sentinel/reports/')
report_dir.mkdir(parents=True, exist_ok=True)

report = f"""# Full Fine-Tuning Report

## Execution Status
- Notebook executed: Yes
- Device: {device}
- GPU: {gpu_name}
- Training config: {TRAINING_CONFIG}

## Dataset
- Path: `{DATASET_ROOT}`
- Manifest: `{MANIFEST}`

## Results
- Checkpoint dir: `{CHECKPOINT_DIR}`
- Latest checkpoint: `{latest if all_checkpoints else 'N/A'}`

## Safety
- Production weights overwritten: No
- Checkpoints saved to Drive: Yes

---
*Report generated: {datetime.now().isoformat()}*
"""

report_path = report_dir / 'full_finetune_report.md'
report_path.write_text(report)

print('Report saved to:', report_path)
print('\nREPORT_CONTENT:')
print('='*60)
print(report)
print('='*60)